In [ ]:
import pandas as pd
import requests
import os
# Questo serve per vedere meglio le tabelle nel notebook
from IPython.display import display 

os.makedirs("documenti_campionati", exist_ok=True)

In [ ]:
# Carichiamo il campione che hai scaricato prima
file_parquet = "dati_epstein/documenti_sample_5000.parquet"
df = pd.read_parquet(file_parquet)

# Visualizziamo i dati in modo elegante
display(df.head(10))

In [ ]:
def download_document(row):
    # Pulizia ID: rimuoviamo estensioni fastidiose (.pdf, .parquet, ecc)
    doc_id_clean = str(row['id']).split('.')[0]
    nome_file = row['original_filename']
    
    # Proviamo prima jdrive, poi files
    endpoints = ["jdrive", "files"]
    
    for ep in endpoints:
        url = f"https://data.jmail.world/v1/{ep}/{doc_id_clean}"
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                with open(f"documenti_campionati/{nome_file}", "wb") as f:
                    f.write(r.content)
                return f"Successo ({ep})"
        except:
            continue
    return "Fallito"

# Testiamo il download sui primi 5 file
df_test = df.head(5).copy()
df_test['status'] = df_test.apply(download_document, axis=1)
display(df_test[['id', 'original_filename', 'status']])